# Exploratory Analysis: Five-Year CO₂ Forecasting

This notebook provides an illustrative, interactive walkthrough of the final analysis dataset and trained model. It is **not** the canonical reproduction path — for that, run `scripts/run_full_pipeline.py` or the individual `src/` modules directly. This notebook is intended for readers who want to inspect intermediate results interactively.

Run this notebook from the repository root after the pipeline has been executed at least once (so that `data/cleaned/ml_co2forecast.csv` and `data/processed/xgb_final.pkl` exist).

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Load the analysis dataset

In [ ]:
df = pd.read_csv('../data/cleaned/ml_co2forecast.csv')
print(f"n={len(df)}, countries={df['country_code'].nunique()}, years={df['year'].min()}-{df['year'].max()}")
df.head()

In [ ]:
df['co2_delta_5yr_pct'].describe()

## 2. Target distribution

In [ ]:
fig, ax = plt.subplots()
ax.hist(df['co2_delta_5yr_pct'], bins=50, color='#2C3E50', alpha=0.8)
ax.axvline(0, color='red', ls='--', lw=1.5)
ax.set_xlabel('5-Year Forward CO₂/Capita % Change')
ax.set_ylabel('Count')
ax.set_title('Target Distribution')
plt.show()

## 3. Load the trained XGBoost model and inspect predictions

In [ ]:
xgb = joblib.load('../data/processed/xgb_final.pkl')
X = np.load('../data/processed/X_scaled.npy')
y = np.load('../data/processed/y.npy')

preds = xgb.predict(X)
print(f"In-sample R²: {r2_score(y, preds):.4f}")
print("(Note: in-sample R² is not the headline result -- see GroupKFold and ",
      "temporal holdout R² in results/model_comparison_*.csv for the ",
      "validated out-of-sample metrics reported in the manuscript.)")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(y, preds, alpha=0.2, s=10, color='#2C3E50')
lims = [min(y.min(), preds.min()), max(y.max(), preds.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual CO₂Δ5yr (%)')
ax.set_ylabel('Predicted CO₂Δ5yr (%)')
ax.legend()
plt.show()

## 4. Explore by income group

In [ ]:
if 'income_groups' in df.columns:
    grouped = df.groupby('income_groups')['co2_delta_5yr_pct'].agg(['mean', 'std', 'count'])
    print(grouped)

## 5. Next steps

For the full validated results (GroupKFold, temporal holdout, LORO, SHAP, ablation, statistical tests), see:

- `results/` -- CSV outputs from each pipeline stage
- `figures/` -- generated figures
- `tables/` -- final manuscript tables
- `manuscript/manuscript.docx` -- the full write-up

To regenerate everything from scratch: `python scripts/run_full_pipeline.py`